# Data Analysis and Modeling

## Installation

Create a virtual environment named `.venv`, activate it, then install the package:

```bash
python -m venv .venv
source .venv/bin/activate # Linux / macOS
python -m pip install --upgrade pip
python -m pip install optimeo
```

After that, launch the app with:

```bash
optimeo
```

If you are using the repo directly, you can also install from GitHub:

```bash
python -m pip install "git+https://github.com/colinbousige/OPTIMEO.git"
```

This notebook shows how to analyze and model experimental data once the package is installed.

In [ ]:
# For Google Colab
!pip install git+https://github.com/colinbousige/OPTIMEO.git

# For local development, prefer:
# uv venv .venv --python 3.10 && source .venv/bin/activate && uv sync

Let's create an `experimental_data(temp, conc)` function that simulates the yield of a chemical reaction based on temperature, concentration A, concentration B and concentration C.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
# pio.renderers.default = "notebook"
pio.renderers.default = "colab"

def experimental_data(temp, cA, cB, cC):
    """
    This function simulates experimental data based on temperature and concentrations.
    The function is not based on any real experimental data and is purely for demonstration purposes.
    """
    out = .2*temp + .5*temp*cA + (cA)/3 + (1 - cB)**2/2 + (3 - cC)/1.5 + np.random.normal(0, 0.2, len(temp))
    return out

def generate_data(N=100):
    temp = np.random.uniform(0, 100, N)
    cA = np.random.uniform(0, 1, N)
    cB = np.random.uniform(0, 1, N)
    cC = np.random.uniform(0, 1, N)
    exp_response = experimental_data(temp, cA, cB, cC)
    # Create a DataFrame with the generated data
    df = pd.DataFrame({'temp': temp, 
                       'cA': cA, 
                       'cB': cB, 
                       'cC': cC, 
                       'response': exp_response})
    return df

df = generate_data(50)
df.to_csv('dataML.csv', index=False)
df.head()

Now, we will use the OPTIMEO package to analyse the data.

In [ ]:
from optimeo.analysis import DataAnalysis

data = pd.read_csv('dataML.csv')
factors = data.columns[:-1].tolist()
response = data.columns[-1]
analysis = DataAnalysis(data, factors, response)
analysis

First, let's take a look at the correlation between the variables.

In [ ]:
analysis.plot_corr()

In [ ]:
analysis.plot_pairplot_plotly()

First, let's look at a simple linear model:

In [ ]:
analysis.compute_linear_model()
analysis.linear_model.summary()

In [ ]:
figs = analysis.plot_linear_model()
for fig in figs:
    fig.show(renderer="colab")

The equation used for the fit is this one, you can change it if you want, e.g [to add interaction terms or other polynomial terms](https://www.statsmodels.org/dev/examples/notebooks/generated/formulas.html):

In [ ]:
analysis.write_equation()

In [ ]:
analysis.equation = 'response ~ temp+ temp:cA + cA + cB + cC'
analysis.compute_linear_model()
analysis.linear_model.summary()

In [ ]:
figs = analysis.plot_linear_model()
for fig in figs:
    fig.show()

Now let's make a ML model to predict the yield based on the temperature and concentrations of A, B and C.

In [ ]:
analysis.model_type = "ElasticNetCV"
# analysis.model_type = "RidgeCV"
# analysis.model_type = "LinearRegression"
# analysis.model_type = "RandomForest"
# analysis.model_type = "GaussianProcess"
# analysis.model_type = "GradientBoosting"
MLmodel = analysis.compute_ML_model()
figs = analysis.plot_ML_model()
for fig in figs:
    fig.show()

And if we want to make a prediction:

In [ ]:
new_value = pd.DataFrame({'temp': [50], 
                          'cA': [0.35], 
                          'cB': [0.5], 
                          'cC': [0.5]})
analysis.predict(new_value)